# DML

## # SCHEMA EVOLUTION 

In [0]:
data = [(2,100,'aa',1),(5,200,'bb',1),(6,300,'cc ',1),(8,400,'dd ',1)]
df = spark.createDataFrame(data,['cust_id','income','name','tip'])

display(df)

df.write.format("delta")\
    .mode("append")\
    .save('/Volumes/databricksnl/default/deltavol/dmlsink/')

cust_id,income,name,tip
2,100,aa,1
5,200,bb,1
6,300,cc,1
8,400,dd,1


In [0]:
%sql
select * from delta.`/Volumes/databricksnl/default/deltavol/dmlsink/`

cust_id,income,name,tip
2,100,aa,1
5,200,bb,1
6,300,cc,1
8,400,dd,1
1,100,aa,1
2,200,bb,1
3,300,cc,1
4,400,dd,1


update

In [0]:
%sql
update delta.`/Volumes/databricksnl/default/deltavol/dmlsink/`
SET income = 1000 WHERE cust_id = 5;

num_affected_rows
1


In [0]:
%sql
select * from delta.`/Volumes/databricksnl/default/deltavol/dmlsink/`

cust_id,income,name,tip
1,100,aa,1
2,200,bb,1
3,300,cc,1
4,400,dd,1
5,1000,bb,1
2,100,aa,1
6,300,cc,1
8,400,dd,1


UPSERT = UPDATE + INSERT

SLOWLY CHANGING DIMENSION

In [0]:
data = [(2,100,'xyz',1),(9,200,'bb',1),(10,300,'cc ',1)]
df = spark.createDataFrame(data,['cust_id','income','name','tip'])

display(df)



cust_id,income,name,tip
2,100,xyz,1
9,200,bb,1
10,300,cc,1


In [0]:
from delta.tables import DeltaTable

In [0]:
dlt_obj = DeltaTable.forPath(spark,"/Volumes/databricksnl/default/deltavol/dmlsink/")

dlt_obj.alias("trg").merge(df.alias("src"), "trg.cust_id = src.cust_id")\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from delta.`/Volumes/databricksnl/default/deltavol/dmlsink/`

cust_id,income,name,tip
1,100,aa,1
3,300,cc,1
4,400,dd,1
5,1000,bb,1
6,300,cc,1
8,400,dd,1
2,100,xyz,1
2,100,xyz,1
9,200,bb,1
10,300,cc,1
